In [33]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [34]:
words= open('names.txt','r').read().splitlines()

In [35]:
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
itos, vocab_size

({1: 'a',
  2: 'b',
  3: 'c',
  4: 'd',
  5: 'e',
  6: 'f',
  7: 'g',
  8: 'h',
  9: 'i',
  10: 'j',
  11: 'k',
  12: 'l',
  13: 'm',
  14: 'n',
  15: 'o',
  16: 'p',
  17: 'q',
  18: 'r',
  19: 's',
  20: 't',
  21: 'u',
  22: 'v',
  23: 'w',
  24: 'x',
  25: 'y',
  26: 'z',
  0: '.'},
 27)

In [36]:
import random
random.seed(42)
random.shuffle(words)

In [37]:
block_size = 3
def build_dataset(words):
    X,y = [],[]
    for w in words:
        context = [0]* block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            y.append(ix)
            context = context[1:]+ [ix]
    X = torch.tensor(X)
    y = torch.tensor(y)
    print(X.shape,y.shape)
    return X,y

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, ytr = build_dataset(words[:n1])
Xdev, ydev = build_dataset(words[n1:n2])
Xte, yte = build_dataset(words[n2:])


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [38]:
for x,y in zip(Xtr[:20], ytr[:20]):
    print("".join(itos[ix.item()] for ix in x), '-->', itos[y.item()])
    

... --> y
..y --> u
.yu --> h
yuh --> e
uhe --> n
hen --> g
eng --> .
... --> d
..d --> i
.di --> o
dio --> n
ion --> d
ond --> r
ndr --> e
dre --> .
... --> x
..x --> a
.xa --> v
xav --> i
avi --> e


In [53]:
class Linear:
    def __init__(self,fan_in,fan_out, bias=True):
        self.weight = torch.randn((fan_in,fan_out))/ fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x@self.weight
        if self.bias is not None:
            self.out+=self.bias
        return self.out
    
    def parameters(self):
        return [self.weight]+ ([] if self.bias is None else [self.bias])

class BatchNorm1D:
    def __init__(self, dim,eps = 1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):
        if self.training:
            xmean = x.mean(0,keepdim=True)
            xvar = x.var(0,keepdim=True)
            with torch.no_grad():
                self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*xmean
                self.running_var = (1-self.momentum)*self.running_var + self.momentum*xvar
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x-xmean)/torch.sqrt(xvar +self.eps)
        self.out = self.gamma*xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []

class Embedding:
    def __init__(self,num_embeddings, embedding_dim):
        self.weight  =torch.randn((num_embeddings, embedding_dim))

    def __call__(self,IX):
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]

class Flatten:
    def __call__(self,x):
        self.out = x.view(x.shape[0],-1)
        return self.out
    
    def parameters(self):
        return []

In [54]:
torch.manual_seed(42)

In [55]:
n_embd = 10
n_hidden=200
# C = torch.randn((vocab_size,n_embd))
layers = [
    Embedding(vocab_size,n_embd), Flatten(),
    Linear(n_embd*block_size, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(),
    Linear(n_hidden,vocab_size)
]

with torch.no_grad():
    layers[-1].weight*=0.1

parameters = [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad=True

12097


In [42]:
max_steps = 200_000
batch_size = 32
lossi=[]

for i in range(max_steps):
    ix = torch.randint(0,Xtr.shape[0],(batch_size,))
    Xb,yb = Xtr[ix],ytr[ix]

    emb = C[Xb]
    x = emb.view(emb.shape[0],-1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,yb)

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i<150000 else 0.01
    for p in parameters:
        p.data-=lr*p.grad

    if i%10000==0:
        print(f"{i}: {loss.item():.4f}")
    lossi.append(loss.log10().item())


0: 3.2966
10000: 2.2322
20000: 2.4111
30000: 2.1004
40000: 2.3157
50000: 2.2104
60000: 1.9653
70000: 1.9767
80000: 2.6738
90000: 2.0837
100000: 2.2730
110000: 1.7491
120000: 2.2891
130000: 2.3443
140000: 2.1731
150000: 1.8246
160000: 1.7614
170000: 2.2418
180000: 2.0803
190000: 2.1326


In [43]:
for layer in layers:
    layer.training=False

In [44]:
@torch.no_grad()
def split_loss(split):
    x,y = {
        'train':[Xtr,ytr],
        "val":[Xdev,ydev],
        'test':[Xte,yte]
    }[split]
    emb = C[x]
    x = emb.view(emb.shape[0],-1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,y)
    print(split, loss.item())

split_loss('train')
split_loss('val')


train 2.0583250522613525
val 2.1065292358398438


In [48]:
for _ in range(20):
    out=[]
    context = [0]*block_size
    while True:
        emb = C[torch.tensor([context])]
        x = emb.view(emb.shape[0],-1)
        for layer in layers:
            x = layer(x)
        logits = x
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:]+[ix]
        out.append(ix)
        if ix==0:
            break
    print(''.join(itos[i] for i in out))

esreezeal.
wyla.
bord.
ser.
apgbe.
saan.
krison.
jevellrah.
rid.
jayden.
layx.
syrias.
cir.
perhan.
yiah.
raelyn.
jacilianan.
calmarian.
giah.
hera.
